# Advanced Python Data Attributes — Problems with Solutions

This notebook is a problem-heavy extension of the lesson on **data attributes**.

## Focus
- class attributes vs. instance attributes
- attribute lookup and shadowing
- `__dict__`
- `getattr`, `setattr`, `hasattr`, `delattr`
- class `mappingproxy`
- mutable class-attribute bugs
- safe shared state
- debugging accidental shadows
- inherited class attributes
- realistic design patterns

Each problem includes a complete solution and executable checks.

> Best study method: predict the output and the relevant `__dict__` contents before running each solution.

## 0. Inspection helpers

These helpers are only for learning/debugging.

In [1]:
from pprint import pprint

def show_state(obj, *names):
    print(f"type={type(obj).__name__}")
    print("instance __dict__ =", getattr(obj, "__dict__", "<no __dict__>"))
    for name in names:
        try:
            value = getattr(obj, name)
        except AttributeError:
            value = "<AttributeError>"
        print(f"{name}: {value!r}")

def section(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

# Problem 1 — Class lookup and instance shadowing

```python
class BankAccount:
    apr = 2.5
```

Create two accounts. Give only the first account its own `apr`, then change the class APR.

### Tasks
1. Predict the final values of `a.apr` and `b.apr`.
2. Predict both instance dictionaries.
3. Explain why the class change reaches only one instance.

In [2]:
class BankAccount:
    apr = 2.5

a = BankAccount()
b = BankAccount()

a.apr = 0.0
BankAccount.apr = 4.25

print("a.apr =", a.apr)
print("b.apr =", b.apr)
print("a.__dict__ =", a.__dict__)
print("b.__dict__ =", b.__dict__)

a.apr = 0.0
b.apr = 4.25
a.__dict__ = {'apr': 0.0}
b.__dict__ = {}


### Solution 1

`a.apr` is `0.0` because `a` has a local instance attribute named `apr`.

`b.apr` is `4.25` because `b` has no local `apr`, so lookup falls back to the class.

Expected state:

```text
a.__dict__ == {'apr': 0.0}
b.__dict__ == {}
BankAccount.__dict__['apr'] == 4.25
```

In [3]:
assert a.apr == 0.0
assert b.apr == 4.25
assert a.__dict__ == {"apr": 0.0}
assert b.__dict__ == {}
assert BankAccount.__dict__["apr"] == 4.25
print("Problem 1 passed.")

Problem 1 passed.


# Problem 2 — Remove a shadow and reveal the class value

A local shadow can be removed.

### Tasks
1. Create `Wallet.currency = "USD"`.
2. Override only `w1.currency` with `"EUR"`.
3. Remove only the instance override.
4. Verify that `w1.currency` becomes `"USD"` again.

In [4]:
class Wallet:
    currency = "USD"

w1 = Wallet()
w2 = Wallet()

w1.currency = "EUR"
print("before:", w1.currency, w2.currency, w1.__dict__)

delattr(w1, "currency")

print("after :", w1.currency, w2.currency, w1.__dict__)
assert w1.currency == "USD"
assert "currency" not in w1.__dict__
assert Wallet.currency == "USD"

before: EUR USD {'currency': 'EUR'}
after : USD USD {}


### Solution 2

`delattr(w1, "currency")` removes the local entry only. The class attribute remains untouched, so ordinary lookup reveals it again.

# Problem 3 — Dynamic fields with `setattr` and `getattr`

Field names may come from parsed input rather than being known in advance.

### Tasks
- create an empty `Profile`,
- add three fields dynamically,
- read them dynamically,
- read a missing field with a default,
- inspect the instance dictionary.

In [5]:
class Profile:
    role = "student"

p = Profile()

updates = {
    "name": "Ada",
    "level": 7,
    "active": True,
}

for field, value in updates.items():
    setattr(p, field, value)

for field in updates:
    print(field, "=>", getattr(p, field))

print("timezone =>", getattr(p, "timezone", "UTC"))
print("role =>", getattr(p, "role"))
print("state =>", p.__dict__)

assert p.__dict__ == updates
assert getattr(p, "timezone", "UTC") == "UTC"
assert p.role == "student"

name => Ada
level => 7
active => True
timezone => UTC
role => student
state => {'name': 'Ada', 'level': 7, 'active': True}


### Solution 3

Use `setattr(obj, dynamic_name, value)` when the name is computed at runtime.

Use `getattr(obj, dynamic_name, default)` when a missing attribute has a legitimate fallback.

# Problem 4 — Mutable class-attribute trap

This class is wrong if each student should have a private course list:

```python
class Student:
    courses = []
```

### Tasks
1. Prove that two instances share the same list.
2. Explain why both instance dictionaries remain empty.
3. Redesign it correctly.

In [6]:
class StudentBuggy:
    courses = []

s1 = StudentBuggy()
s2 = StudentBuggy()

s1.courses.append("Python")

print("s1.courses =", s1.courses)
print("s2.courses =", s2.courses)
print("same list?  ", s1.courses is s2.courses)
print("s1 state    ", s1.__dict__)
print("s2 state    ", s2.__dict__)

s1.courses = ['Python']
s2.courses = ['Python']
same list?   True
s1 state     {}
s2 state     {}


### Solution 4

Appending mutates the shared class-level list itself. It does **not** assign a new instance attribute, so both instance dictionaries can remain empty.

Per-instance mutable state belongs in `__init__`.

In [7]:
class Student:
    school = "Open Academy"

    def __init__(self, name):
        self.name = name
        self.courses = []

s1 = Student("Ada")
s2 = Student("Grace")

s1.courses.append("Python")

print(s1.__dict__)
print(s2.__dict__)
print("same list?", s1.courses is s2.courses)

assert s1.courses == ["Python"]
assert s2.courses == []
assert s1.courses is not s2.courses

{'name': 'Ada', 'courses': ['Python']}
{'name': 'Grace', 'courses': []}
same list? False


# Problem 5 — Shared default with an optional per-instance override

Design `ServiceClient` so that:
- the default timeout is shared at class level,
- each client always has a name,
- only clients with custom timeouts store `timeout` locally,
- an override can be reset.

### Solution 5

In [8]:
class ServiceClient:
    timeout = 30

    def __init__(self, name, timeout=None):
        self.name = name
        if timeout is not None:
            self.timeout = timeout

    def reset_timeout(self):
        if "timeout" in self.__dict__:
            del self.timeout

normal = ServiceClient("normal")
fast = ServiceClient("fast", timeout=5)

print(normal.__dict__)
print(fast.__dict__)
print(normal.timeout, fast.timeout)

ServiceClient.timeout = 45
print("after class change:", normal.timeout, fast.timeout)

fast.reset_timeout()
print("after reset:", normal.timeout, fast.timeout)

assert normal.timeout == 45
assert fast.timeout == 45
assert "timeout" not in fast.__dict__

{'name': 'normal'}
{'name': 'fast', 'timeout': 5}
30 5
after class change: 45 5
after reset: 45 45


# Problem 6 — Accidental shadowing of a shared counter

Why is this buggy?

```python
class Job:
    completed = 0

    def mark_complete(self):
        self.completed += 1
```

### Tasks
1. Create two jobs and call the method.
2. Inspect the class counter and instance dictionaries.
3. Fix the implementation.

In [9]:
class JobBuggy:
    completed = 0

    def mark_complete(self):
        self.completed += 1

j1 = JobBuggy()
j2 = JobBuggy()

j1.mark_complete()
j2.mark_complete()

print("class:", JobBuggy.completed)
print("j1:", j1.completed, j1.__dict__)
print("j2:", j2.completed, j2.__dict__)

class: 0
j1: 1 {'completed': 1}
j2: 1 {'completed': 1}


### Solution 6

`self.completed += 1` reads the visible value, creates a new integer, and assigns that integer back to `self.completed`. The assignment creates an instance attribute.

If the state is intentionally shared, update the class.

In [10]:
class Job:
    completed = 0

    def mark_complete(self):
        type(self).completed += 1

j1 = Job()
j2 = Job()

j1.mark_complete()
j2.mark_complete()
j1.mark_complete()

print(Job.completed)
print(j1.__dict__)
print(j2.__dict__)

assert Job.completed == 3
assert "completed" not in j1.__dict__
assert "completed" not in j2.__dict__

3
{}
{}


# Problem 7 — Explore the class `mappingproxy`

A class `__dict__` is exposed through a `mappingproxy`.

### Tasks
1. Print its type.
2. Try item assignment.
3. Catch the exception.
4. Add the class attribute correctly.

In [11]:
class Config:
    mode = "safe"

print(type(Config.__dict__))
print(Config.__dict__["mode"])

try:
    Config.__dict__["retries"] = 3
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)

Config.retries = 3
print(Config.retries)
print("stored on class?", "retries" in Config.__dict__)

assert Config.retries == 3

<class 'mappingproxy'>
safe
TypeError: 'mappingproxy' object does not support item assignment
3
stored on class? True


### Solution 7

Treat `Class.__dict__` as an inspection view. Change class state through attribute operations:

```python
ClassName.x = value
setattr(ClassName, "x", value)
delattr(ClassName, "x")
```

# Problem 8 — Direct instance `__dict__` mutation

For ordinary objects with an instance dictionary, compare:

```python
obj.version = "3.12"
obj.__dict__["version"] = "3.12"
setattr(obj, "version", "3.12")
```

### Task
Demonstrate all three, then choose the preferred style for normal code.

In [12]:
class Runtime:
    language = "Python"

r = Runtime()

r.version = "3.11"
print(r.__dict__)

r.__dict__["version"] = "3.12"
print(r.version)

setattr(r, "version", "3.13")
print(r.version)
print(r.__dict__)

assert r.version == "3.13"

{'version': '3.11'}
3.12
3.13
{'version': '3.13'}


### Solution 8

- Use dotted assignment when the name is known.
- Use `setattr` when the name is dynamic.
- Prefer direct `__dict__` access mainly for introspection, debugging, controlled serialization, or low-level tooling.

# Problem 9 — Remove redundant shadows

Write `remove_redundant_shadow(obj, name)`.

It should remove a local attribute only when:
- it exists in the instance dictionary,
- the class has an attribute with that name,
- the local value equals the class value.

Return `True` when a shadow is removed.

In [13]:
class Feature:
    enabled = True

def remove_redundant_shadow(obj, name):
    state = obj.__dict__

    if name not in state:
        return False

    cls = type(obj)

    if not hasattr(cls, name):
        return False

    if state[name] == getattr(cls, name):
        delattr(obj, name)
        return True

    return False

f1 = Feature()
f2 = Feature()
f3 = Feature()

f1.enabled = True
f2.enabled = False

print(remove_redundant_shadow(f1, "enabled"))
print(remove_redundant_shadow(f2, "enabled"))
print(remove_redundant_shadow(f3, "enabled"))

print(f1.__dict__, f2.__dict__, f3.__dict__)

assert f1.__dict__ == {}
assert f2.__dict__ == {"enabled": False}
assert f3.__dict__ == {}

True
False
False
{} {'enabled': False} {}


### Solution 9

This is useful when an instance has accidentally copied a class default into local state. Removing a redundant shadow lets the instance follow future class-level changes again.

# Problem 10 — Safe bulk updates with an allow-list

Blindly calling `setattr` for every external field can create unwanted attributes.

Write `apply_updates(obj, updates, allowed)` that:
- applies allowed fields,
- rejects all other fields,
- returns rejected fields in a dictionary.

In [14]:
class AppSettings:
    theme = "light"
    language = "en"

def apply_updates(obj, updates, allowed):
    rejected = {}

    for name, value in updates.items():
        if name in allowed:
            setattr(obj, name, value)
        else:
            rejected[name] = value

    return rejected

settings = AppSettings()

incoming = {
    "theme": "dark",
    "language": "bg",
    "admin": True,
    "debug_secret": "do-not-store",
}

rejected = apply_updates(settings, incoming, {"theme", "language"})

print(settings.__dict__)
print(rejected)

assert settings.theme == "dark"
assert settings.language == "bg"
assert not hasattr(settings, "admin")
assert "debug_secret" not in settings.__dict__

{'theme': 'dark', 'language': 'bg'}
{'admin': True, 'debug_secret': 'do-not-store'}


### Solution 10

Dynamic attributes are powerful, so production code should usually validate field names before assigning them.

# Problem 11 — Inherited class attributes

Study this sequence:

```python
class Base:
    rate = 10

class Premium(Base):
    pass
```

Then:
1. create an instance,
2. change `Base.rate`,
3. add `Premium.rate`,
4. add `p.rate`,
5. delete `p.rate`,
6. delete `Premium.rate`.

Predict the visible value at every step.

In [15]:
class Base:
    rate = 10

class Premium(Base):
    pass

p = Premium()
print("1:", p.rate)

Base.rate = 20
print("2:", p.rate)

Premium.rate = 30
print("3:", p.rate)

p.rate = 40
print("4:", p.rate)

del p.rate
print("5:", p.rate)

del Premium.rate
print("6:", p.rate)

assert p.rate == 20
assert "rate" not in p.__dict__
assert "rate" not in Premium.__dict__
assert Base.__dict__["rate"] == 20

1: 10
2: 20
3: 30
4: 40
5: 30
6: 20


### Solution 11

For this ordinary attribute example, a useful mental model is:

```text
instance -> subclass -> base class(es) -> AttributeError
```

Earlier matches hide later ones.

# Problem 12 — Existing instances observe later class changes

Prove that instances do not automatically copy every class attribute at creation time.

### Tasks
- create two objects,
- change an existing class attribute afterward,
- add a brand-new class attribute afterward,
- verify both existing objects can see the changes while their instance dictionaries remain empty.

In [16]:
class Device:
    status = "offline"

d1 = Device()
d2 = Device()

print("initial:", d1.status, d2.status)

Device.status = "online"
Device.region = "EU"

print("later:", d1.status, d2.status)
print("region:", d1.region, d2.region)
print(d1.__dict__, d2.__dict__)

assert d1.__dict__ == {}
assert d2.__dict__ == {}
assert d1.status == "online"
assert d2.region == "EU"

initial: offline offline
later: online online
region: EU EU
{} {}


### Solution 12

Because no local shadow exists, lookup continues to the current class namespace each time.

# Problem 13 — Build an attribute-debugging report

Write `attribute_report(obj, name)` that returns:
- whether the name exists in `obj.__dict__`,
- its local value,
- whether the name exists directly in `type(obj).__dict__`,
- the direct class value,
- the finally resolved value,
- whether local state shadows the immediate class.

For this exercise, do not recursively inspect base classes.

In [17]:
def attribute_report(obj, name):
    instance_dict = getattr(obj, "__dict__", {})
    class_dict = type(obj).__dict__

    local_exists = name in instance_dict
    class_exists = name in class_dict

    return {
        "attribute": name,
        "local_exists": local_exists,
        "local_value": instance_dict.get(name),
        "class_exists": class_exists,
        "class_value": class_dict.get(name),
        "resolved_value": getattr(obj, name, "<missing>"),
        "is_shadowing": local_exists and class_exists,
    }

class Account:
    apr = 2.5
    account_type = "Savings"

x = Account()
x.apr = 0.0
x.owner = "Ada"

for field in ("apr", "account_type", "owner", "missing"):
    section(field)
    pprint(attribute_report(x, field))

assert attribute_report(x, "apr")["is_shadowing"] is True
assert attribute_report(x, "owner")["class_exists"] is False


apr
{'attribute': 'apr',
 'class_exists': True,
 'class_value': 2.5,
 'is_shadowing': True,
 'local_exists': True,
 'local_value': 0.0,
 'resolved_value': 0.0}

account_type
{'attribute': 'account_type',
 'class_exists': True,
 'class_value': 'Savings',
 'is_shadowing': False,
 'local_exists': False,
 'local_value': None,
 'resolved_value': 'Savings'}

owner
{'attribute': 'owner',
 'class_exists': False,
 'class_value': None,
 'is_shadowing': False,
 'local_exists': True,
 'local_value': 'Ada',
 'resolved_value': 'Ada'}

missing
{'attribute': 'missing',
 'class_exists': False,
 'class_value': None,
 'is_shadowing': False,
 'local_exists': False,
 'local_value': None,
 'resolved_value': '<missing>'}


### Solution 13

This kind of report is useful when an attribute appears to ignore a class-level change. The first thing to check is whether an instance-level shadow exists.

# Problem 14 — Repair a mixed-state design

The intended behavior of an API client is:

- `headers`: unique per client,
- `timeout`: shared default with optional per-client override,
- `request_count`: truly shared across clients.

Rewrite the class correctly.

In [18]:
class APIClient:
    timeout = 30
    request_count = 0

    def __init__(self, name):
        self.name = name
        self.headers = {}

    def set_header(self, key, value):
        self.headers[key] = value

    def set_timeout(self, seconds):
        self.timeout = seconds

    def reset_timeout(self):
        if "timeout" in self.__dict__:
            del self.timeout

    def record_request(self):
        type(self).request_count += 1

c1 = APIClient("alpha")
c2 = APIClient("beta")

c1.set_header("Authorization", "token-A")
c2.set_header("Authorization", "token-B")
c1.set_timeout(5)

c1.record_request()
c2.record_request()
c2.record_request()

print("c1 headers:", c1.headers)
print("c2 headers:", c2.headers)
print("timeouts:", c1.timeout, c2.timeout)
print("request_count:", APIClient.request_count)
print("c1 state:", c1.__dict__)
print("c2 state:", c2.__dict__)

assert c1.headers is not c2.headers
assert c1.timeout == 5
assert c2.timeout == 30
assert APIClient.request_count == 3
assert "request_count" not in c1.__dict__

c1 headers: {'Authorization': 'token-A'}
c2 headers: {'Authorization': 'token-B'}
timeouts: 5 30
request_count: 3
c1 state: {'name': 'alpha', 'headers': {'Authorization': 'token-A'}, 'timeout': 5}
c2 state: {'name': 'beta', 'headers': {'Authorization': 'token-B'}}


### Solution 14

This pattern separates:
- **shared configuration** (`timeout`),
- **shared aggregate state** (`request_count`),
- **object-specific mutable state** (`headers`),
- **object-specific identity/state** (`name`).

# Problem 15 — Capstone: bank-wide APR plus per-account overrides

Build a realistic account model.

### Requirements
- `BankAccount.apr` is the bank-wide default.
- Every account stores `owner` and `balance`.
- An account may optionally shadow `apr`.
- `effective_apr()` returns the visible APR.
- `has_custom_apr()` detects a local shadow.
- `reset_apr()` removes only the local shadow.
- Later class-level APR changes affect all accounts without overrides.

In [19]:
class BankAccount:
    apr = 2.5
    account_type = "Savings"

    def __init__(self, owner, balance=0.0, apr_override=None):
        self.owner = owner
        self.balance = float(balance)

        if apr_override is not None:
            self.apr = float(apr_override)

    def effective_apr(self):
        return self.apr

    def has_custom_apr(self):
        return "apr" in self.__dict__

    def set_apr(self, value):
        self.apr = float(value)

    def reset_apr(self):
        if "apr" in self.__dict__:
            del self.apr

    def state(self):
        return {
            "owner": self.owner,
            "balance": self.balance,
            "effective_apr": self.effective_apr(),
            "custom_apr": self.has_custom_apr(),
        }

ada = BankAccount("Ada", 1000)
grace = BankAccount("Grace", 1500, apr_override=1.75)
linus = BankAccount("Linus", 500)

print("Initial:")
pprint(ada.state())
pprint(grace.state())
pprint(linus.state())

BankAccount.apr = 3.1

print("\nAfter bank-wide APR change:")
pprint(ada.state())
pprint(grace.state())
pprint(linus.state())

grace.reset_apr()

print("\nAfter reset:")
pprint(grace.state())

assert ada.effective_apr() == 3.1
assert linus.effective_apr() == 3.1
assert grace.effective_apr() == 3.1
assert not grace.has_custom_apr()

Initial:
{'balance': 1000.0, 'custom_apr': False, 'effective_apr': 2.5, 'owner': 'Ada'}
{'balance': 1500.0, 'custom_apr': True, 'effective_apr': 1.75, 'owner': 'Grace'}
{'balance': 500.0, 'custom_apr': False, 'effective_apr': 2.5, 'owner': 'Linus'}

After bank-wide APR change:
{'balance': 1000.0, 'custom_apr': False, 'effective_apr': 3.1, 'owner': 'Ada'}
{'balance': 1500.0, 'custom_apr': True, 'effective_apr': 1.75, 'owner': 'Grace'}
{'balance': 500.0, 'custom_apr': False, 'effective_apr': 3.1, 'owner': 'Linus'}

After reset:
{'balance': 1500.0, 'custom_apr': False, 'effective_apr': 3.1, 'owner': 'Grace'}


### Solution 15

The class default is stored only once. A custom APR exists only on accounts that need it. Removing the override immediately reconnects the instance to the bank-wide default.

# Problem 16 — Predict-before-running challenge

Predict the **exact** output before running.

In [20]:
class A:
    x = 1

a1 = A()
a2 = A()

a1.x = A.x + 10
A.x += 100
a2.x = a1.x + A.x

print("A.x =", A.x)
print("a1.x =", a1.x)
print("a2.x =", a2.x)
print("a1.__dict__ =", a1.__dict__)
print("a2.__dict__ =", a2.__dict__)

A.x = 101
a1.x = 11
a2.x = 112
a1.__dict__ = {'x': 11}
a2.__dict__ = {'x': 112}


### Solution 16

Step by step:

1. `A.x == 1`
2. `a1.x = 11` creates an instance shadow.
3. `A.x += 100` changes only the class value to `101`.
4. `a2.x = 11 + 101` creates `a2.x == 112`.

Expected final state:

```text
A.x = 101
a1.x = 11
a2.x = 112
a1.__dict__ = {'x': 11}
a2.__dict__ = {'x': 112}
```

In [21]:
assert A.x == 101
assert a1.x == 11
assert a2.x == 112
assert a1.__dict__ == {"x": 11}
assert a2.__dict__ == {"x": 112}
print("Prediction verified.")

Prediction verified.


# Problem 17 — Refactor repeated defaults into class attributes

Start with a design where every report stores identical default values locally.

Refactor so:
- `title` stays per-instance,
- `format`, `language`, and `page_size` are class defaults,
- one report can override only `language`,
- later class-level changes affect non-overridden objects.

In [22]:
class Report:
    format = "pdf"
    language = "en"
    page_size = "A4"

    def __init__(self, title):
        self.title = title

r1 = Report("Quarterly Results")
r2 = Report("Annual Review")

r2.language = "fr"

print("before:")
print(r1.format, r1.language, r1.page_size)
print(r2.format, r2.language, r2.page_size)

Report.format = "html"

print("\nafter:")
print(r1.format, r1.language, r1.page_size)
print(r2.format, r2.language, r2.page_size)

print("\nstate:")
print(r1.__dict__)
print(r2.__dict__)

assert r1.__dict__ == {"title": "Quarterly Results"}
assert r2.__dict__ == {"title": "Annual Review", "language": "fr"}
assert r1.format == "html"
assert r2.format == "html"

before:
pdf en A4
pdf fr A4

after:
html en A4
html fr A4

state:
{'title': 'Quarterly Results'}
{'title': 'Annual Review', 'language': 'fr'}


### Solution 17

This is a clean use of class attributes as defaults: they are shared until an instance deliberately overrides one.

# Problem 18 — Audit many objects for overrides

Write `find_overrides(objects, name)`.

Return `(object, local_value)` pairs only for objects whose **instance dictionary** contains `name`.

Use it to find accounts with custom APR values.

In [23]:
def find_overrides(objects, name):
    result = []
    for obj in objects:
        state = getattr(obj, "__dict__", {})
        if name in state:
            result.append((obj, state[name]))
    return result

BankAccount.apr = 4.0

accounts = [
    BankAccount("A", 100),
    BankAccount("B", 200, apr_override=3.5),
    BankAccount("C", 300),
    BankAccount("D", 400, apr_override=2.9),
]

overrides = find_overrides(accounts, "apr")

for account, local_apr in overrides:
    print(account.owner, "=>", local_apr)

assert [(account.owner, rate) for account, rate in overrides] == [
    ("B", 3.5),
    ("D", 2.9),
]

B => 3.5
D => 2.9


# Problem 19 — Normalize stale overrides

Some objects have local values that now equal the current class/default value.

Write `normalize_overrides(objects, name)` that:
- removes only redundant local overrides,
- preserves meaningful overrides,
- returns the number removed.

This solution deliberately removes the local value temporarily to discover what normal lookup would reveal.

In [24]:
def normalize_overrides(objects, name):
    removed = 0

    for obj in objects:
        state = getattr(obj, "__dict__", {})
        if name not in state:
            continue

        local_value = state[name]
        delattr(obj, name)

        try:
            fallback_value = getattr(obj, name)
        except AttributeError:
            setattr(obj, name, local_value)
            continue

        if local_value == fallback_value:
            removed += 1
        else:
            setattr(obj, name, local_value)

    return removed

class Worker:
    region = "EU"

workers = [Worker() for _ in range(4)]
workers[0].region = "EU"
workers[1].region = "US"
workers[2].region = "EU"

removed = normalize_overrides(workers, "region")

print("removed:", removed)
for index, worker in enumerate(workers):
    print(index, worker.region, worker.__dict__)

assert removed == 2
assert workers[0].__dict__ == {}
assert workers[1].__dict__ == {"region": "US"}
assert workers[2].__dict__ == {}
assert workers[3].__dict__ == {}

removed: 2
0 EU {}
1 US {'region': 'US'}
2 EU {}
3 EU {}


# Problem 20 — Class mutation through `setattr`

Dynamic configuration can target the class itself.

### Tasks
- start with a class default `mode="safe"`,
- dynamically add/update class settings from a dictionary,
- verify old instances see the new class values,
- then override one setting on one instance.

In [25]:
class Engine:
    mode = "safe"

e1 = Engine()
e2 = Engine()

class_updates = {
    "mode": "fast",
    "retries": 4,
    "region": "EU",
}

for name, value in class_updates.items():
    setattr(Engine, name, value)

print(e1.mode, e1.retries, e1.region)
print(e2.mode, e2.retries, e2.region)

e1.mode = "diagnostic"

print("after e1 override:")
print(e1.mode, e2.mode)
print("e1 state:", e1.__dict__)
print("class mode:", Engine.mode)

assert e1.mode == "diagnostic"
assert e2.mode == "fast"
assert Engine.mode == "fast"

fast 4 EU
fast 4 EU
after e1 override:
diagnostic fast
e1 state: {'mode': 'diagnostic'}
class mode: fast


# Problem 21 — Detect whether a value is local or inherited

Write a helper `attribute_origin(obj, name)` that returns:

- `"instance"` if the name is in `obj.__dict__`,
- `"class"` if it is directly in `type(obj).__dict__`,
- `"inherited"` if it is available through lookup but not in either immediate dictionary,
- `"missing"` otherwise.

In [26]:
def attribute_origin(obj, name):
    if name in getattr(obj, "__dict__", {}):
        return "instance"

    if name in type(obj).__dict__:
        return "class"

    if hasattr(obj, name):
        return "inherited"

    return "missing"

class Parent:
    inherited_value = 10

class Child(Parent):
    own_class_value = 20

child = Child()
child.local_value = 30

print(attribute_origin(child, "local_value"))
print(attribute_origin(child, "own_class_value"))
print(attribute_origin(child, "inherited_value"))
print(attribute_origin(child, "does_not_exist"))

assert attribute_origin(child, "local_value") == "instance"
assert attribute_origin(child, "own_class_value") == "class"
assert attribute_origin(child, "inherited_value") == "inherited"
assert attribute_origin(child, "does_not_exist") == "missing"

instance
class
inherited
missing


# Problem 22 — Final integrated debugging scenario

A team reports:

> "We changed the class configuration, but a few objects still show the old value."

Build a small diagnostic that:
1. prints the current class value,
2. finds every instance that locally shadows the name,
3. prints the shadowing values,
4. optionally removes only shadows equal to the class value,
5. confirms which instances still intentionally differ.

In [27]:
class Node:
    region = "EU"

nodes = [Node() for _ in range(5)]

nodes[0].region = "US"
nodes[1].region = "EU"
nodes[2].region = "APAC"
# nodes[3] and nodes[4] have no local override

print("class region:", Node.region)

print("\nBefore cleanup:")
for i, node in enumerate(nodes):
    print(i, "visible=", node.region, "local=", node.__dict__.get("region", "<none>"))

removed = normalize_overrides(nodes, "region")

print("\nRemoved redundant overrides:", removed)
print("\nAfter cleanup:")
for i, node in enumerate(nodes):
    print(i, "visible=", node.region, "local=", node.__dict__.get("region", "<none>"))

assert removed == 1
assert nodes[0].region == "US"
assert nodes[2].region == "APAC"
assert "region" not in nodes[1].__dict__

class region: EU

Before cleanup:
0 visible= US local= US
1 visible= EU local= EU
2 visible= APAC local= APAC
3 visible= EU local= <none>
4 visible= EU local= <none>

Removed redundant overrides: 1

After cleanup:
0 visible= US local= US
1 visible= EU local= <none>
2 visible= APAC local= APAC
3 visible= EU local= <none>
4 visible= EU local= <none>


# Final review — Best practices

1. Use **class attributes** for intentionally shared defaults, constants, and shared state.
2. Use **instance attributes** for per-object state.
3. Initialize per-instance mutable objects such as lists and dictionaries inside `__init__`.
4. Remember that assignment through an instance can create a **shadow**.
5. If a class change appears to have no effect, inspect `obj.__dict__` for a same-named local value.
6. Delete a local shadow when you want the object to follow the class value again.
7. Use `getattr`/`setattr` for genuinely dynamic names.
8. Validate dynamic field names before assigning external data.
9. Prefer `obj.x = value` to direct `obj.__dict__` mutation in ordinary code.
10. Treat `Class.__dict__` as a `mappingproxy` inspection view.
11. Be especially careful with `self.shared_counter += 1`; it may create local state.
12. Existing instances can observe class attributes changed or added later if they do not shadow them.
13. In inheritance, a subclass attribute can hide a base-class attribute, and an instance attribute can hide both.
14. Use assertions and state inspection to make attribute behavior mechanically testable.

## Practice loop

For every exercise:
- predict the visible attribute value,
- predict which namespace owns the value,
- inspect `obj.__dict__`,
- inspect `type(obj).__dict__`,
- explain the lookup in one sentence.